# 📦 01 — Data Collection Pipeline
### Forecasting & Identifying Global Export Opportunities for Algerian Exporters
**ENSIA — Machine Learning Project | Spring 2025–2026**

---

This notebook collects **all raw data** needed for the project from 4 public APIs.

| # | Table | Source | Description |
|---|-------|--------|-------------|
| 1 | `01_comtrade_world_imports.csv` | UN Comtrade | Global imports — who buys what product from whom |
| 2 | `02_algeria_resources_fao.csv` | FAO FAOSTAT | Algeria production, area harvested, yield |
| 3 | `03_algeria_resources_fao_inputs.csv` | FAO FAOSTAT | Algeria fertilizer & input use |
| 4 | `04_algeria_resources_fao_land.csv` | FAO FAOSTAT | Algeria land use & irrigation |
| 5 | `05_algeria_resources_fao_prices.csv` | FAO FAOSTAT | Algeria domestic producer prices |
| 6 | `06_algeria_resources_worldbank.csv` | World Bank | Algeria resource rents & indicators |
| 7 | `07_fao_commodity_prices_monthly.csv` | FAO FAOSTAT | World commodity prices (monthly) |
| 8 | `08_worldbank_commodity_index.csv` | World Bank | Global commodity price index |
| 9 | `09_comtrade_unit_values.csv` | Derived | Implicit price per kg (value ÷ weight) |
| 10 | `10_algeria_imports_comtrade.csv` | UN Comtrade | What Algeria buys = supply gaps |
| 11 | `11_country_metadata.csv` | World Bank | GDP, population, trade for all countries |
| 12 | `12_algeria_exports_comtrade.csv` | OEC / BACI | Algeria actual exports by product & destination |

**All files are saved to `data/raw/`** — matching the repository structure:
```
algerian-export-opportunities/
└── data/
    ├── raw/         ← output of this notebook (Tables 1–12)
    └── processed/   ← output of next notebook (02_data_preparation_eda.ipynb)
```


---
## ⚙️ 0 — Setup
Install required packages and create the folder structure.

In [ ]:
pip install requests pandas numpy scikit-learn

In [ ]:
# Create repository folder structure
# This matches:  algerian-export-opportunities/data/raw  and  data/processed
from pathlib import Path
import logging, json, time, os, requests, zipfile, io
import pandas as pd
import numpy as np
from datetime import datetime

for folder in ["data/raw", "data/processed", "data/cache", "logs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(f"logs/collection_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)
print("✅ Folders created:  data/raw/  |  data/processed/  |  data/cache/  |  logs/")


---
## 📋 Tables 1–11 — Full Data Collection Pipeline
This cell contains the complete pipeline from Pipeline v1 (overview) plus Pipeline v2 (detailed collection).

### What Pipeline v2 collects:
- **Table 1**: UN Comtrade — all world bilateral import flows (each country importing each HS product from each partner, 2018–2022)
- **Tables 2–5**: FAO FAOSTAT — Algeria production, inputs, land use, prices
- **Tables 6, 11**: World Bank — Algeria resource indicators + all-country metadata
- **Tables 7–8**: Global commodity prices (FAO + World Bank)
- **Table 9**: Derived — implicit unit price per kg from Table 1
- **Table 10**: UN Comtrade — Algeria own imports (reveals supply gaps)

> ⏱️ **Runtime note:** Table 1 makes ~150 API calls (30 products × 5 years). With the 1.5s delay per call, expect ~5–10 minutes. All results are cached in `data/cache/` — re-running is instant.


In [ ]:
"""
=============================================================================
ALGERIA TRADE & RESOURCES — FULL GRANULAR DATA COLLECTION PIPELINE
=============================================================================

Tables produced:
  data/raw/
    01_comtrade_world_imports.csv       ← ALL countries importing each product,
                                           per year 2018-2022, from EACH partner
    02_algeria_resources_fao.csv        ← FAO: production, area, yield, stocks
    03_algeria_resources_fao_inputs.csv ← FAO: fertilizers, pesticides, machinery
    04_algeria_resources_fao_land.csv   ← FAO: land use, irrigation, forest
    05_algeria_resources_fao_prices.csv ← FAO: domestic producer prices
    06_algeria_resources_worldbank.csv  ← World Bank: resource rents, reserves
    07_fao_commodity_prices_monthly.csv ← FAO: world commodity prices monthly
    08_worldbank_commodity_index.csv    ← WB: commodity price indices
    09_comtrade_unit_values.csv         ← Comtrade: value/kg (implicit price)
    10_algeria_imports_comtrade.csv     ← What Algeria buys (= what it lacks)
    11_country_metadata.csv            ← GDP, pop, trade openness per country

All columns kept exactly as returned by the API + enrichment columns added.
NO aggregation. NO scoring. Pure raw data.
=============================================================================
"""

import requests
import pandas as pd
import time
import json
import os
import logging
import traceback
from pathlib import Path
from datetime import datetime

# ─── Logging ──────────────────────────────────────────────────────────────────
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/cache").mkdir(parents=True, exist_ok=True)
Path("logs").mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(f"logs/collection_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─── Progress Tracker ─────────────────────────────────────────────────────────
# Saves which calls completed so we can RESUME if interrupted
PROGRESS_FILE = "data/cache/progress.json"

def load_progress():
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"completed": []}

def mark_done(key):
    p = load_progress()
    if key not in p["completed"]:
        p["completed"].append(key)
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f)

def is_done(key):
    return key in load_progress()["completed"]

# ─── Configuration ────────────────────────────────────────────────────────────

YEARS = ["2018", "2019", "2020", "2021", "2022"]   # 5-year trend

# HS codes to collect — expanded, all Algeria-relevant sectors
# Format: hs_code_6: { name, chapter, sector, algeria_has_resource }
HS_PRODUCTS = {
    # ── Dates & Tree Fruits ──
    "080410": {"name": "Dates fresh or dried",              "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080290": {"name": "Other nuts (pistachios almonds)",   "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080610": {"name": "Grapes fresh",                      "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080510": {"name": "Oranges fresh/dried",               "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080520": {"name": "Mandarins clementines",             "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080550": {"name": "Lemons limes",                      "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": True},
    "080440": {"name": "Avocados",                          "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": False},
    "080430": {"name": "Pineapples",                        "chapter": "08", "sector": "fruits_nuts",   "algeria_resource": False},
    # ── Vegetables ──
    "070200": {"name": "Tomatoes fresh/chilled",            "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070700": {"name": "Cucumbers gherkins fresh",          "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070930": {"name": "Aubergines eggplant fresh",         "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070960": {"name": "Chillies peppers fresh",            "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070190": {"name": "Potatoes fresh not seed",           "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070310": {"name": "Onions shallots fresh",             "chapter": "07", "sector": "vegetables",    "algeria_resource": True},
    "070920": {"name": "Asparagus fresh",                   "chapter": "07", "sector": "vegetables",    "algeria_resource": False},
    # ── Cereals ──
    "100190": {"name": "Wheat durum and other",             "chapter": "10", "sector": "cereals",       "algeria_resource": True},
    "100300": {"name": "Barley",                            "chapter": "10", "sector": "cereals",       "algeria_resource": True},
    "100590": {"name": "Maize corn not seed",               "chapter": "10", "sector": "cereals",       "algeria_resource": True},
    "100630": {"name": "Semi-milled or wholly milled rice", "chapter": "10", "sector": "cereals",       "algeria_resource": False},
    # ── Oils & Oilseeds ──
    "151110": {"name": "Palm oil crude",                    "chapter": "15", "sector": "oils",          "algeria_resource": False},
    "151190": {"name": "Palm oil refined",                  "chapter": "15", "sector": "oils",          "algeria_resource": False},
    "150910": {"name": "Olive oil virgin",                  "chapter": "15", "sector": "oils",          "algeria_resource": True},
    "150990": {"name": "Olive oil other",                   "chapter": "15", "sector": "oils",          "algeria_resource": True},
    "120100": {"name": "Soya beans",                        "chapter": "12", "sector": "oilseeds",      "algeria_resource": False},
    "120600": {"name": "Sunflower seeds",                   "chapter": "12", "sector": "oilseeds",      "algeria_resource": True},
    # ── Livestock & Dairy ──
    "020110": {"name": "Bovine carcasses meat fresh",       "chapter": "02", "sector": "livestock",     "algeria_resource": True},
    "020410": {"name": "Lamb carcasses meat fresh",         "chapter": "02", "sector": "livestock",     "algeria_resource": True},
    "040110": {"name": "Milk cream not concentrated",       "chapter": "04", "sector": "dairy",         "algeria_resource": True},
    "040210": {"name": "Milk powder",                       "chapter": "04", "sector": "dairy",         "algeria_resource": True},
    "040690": {"name": "Cheese other",                      "chapter": "04", "sector": "dairy",         "algeria_resource": False},
    # ── Fish & Seafood ──
    "030269": {"name": "Other fish fresh not fillets",      "chapter": "03", "sector": "fisheries",     "algeria_resource": True},
    "030379": {"name": "Other fish frozen",                 "chapter": "03", "sector": "fisheries",     "algeria_resource": True},
    "030612": {"name": "Frozen rock lobster",               "chapter": "03", "sector": "fisheries",     "algeria_resource": True},
    # ── Energy ──
    "270900": {"name": "Crude petroleum oils",              "chapter": "27", "sector": "energy",        "algeria_resource": True},
    "271111": {"name": "LNG liquefied natural gas",         "chapter": "27", "sector": "energy",        "algeria_resource": True},
    "271121": {"name": "Natural gas gaseous state",         "chapter": "27", "sector": "energy",        "algeria_resource": True},
    "271019": {"name": "Petroleum oils refined other",      "chapter": "27", "sector": "energy",        "algeria_resource": True},
    "271600": {"name": "Electrical energy",                 "chapter": "27", "sector": "energy",        "algeria_resource": True},
    # ── Minerals & Mining ──
    "260111": {"name": "Iron ores non-agglomerated",        "chapter": "26", "sector": "minerals",      "algeria_resource": True},
    "260300": {"name": "Copper ores concentrates",          "chapter": "26", "sector": "minerals",      "algeria_resource": True},
    "260200": {"name": "Manganese ores",                    "chapter": "26", "sector": "minerals",      "algeria_resource": True},
    "251010": {"name": "Natural calcium phosphates",        "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    "250900": {"name": "Chalk",                             "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    "252310": {"name": "Cement clinkers",                   "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    "252329": {"name": "Portland cement other",             "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    "270310": {"name": "Peat not agglomerated",             "chapter": "27", "sector": "minerals",      "algeria_resource": True},
    "710812": {"name": "Gold non-monetary unwrought",       "chapter": "71", "sector": "minerals",      "algeria_resource": True},
    "261610": {"name": "Silver ores concentrates",          "chapter": "26", "sector": "minerals",      "algeria_resource": True},
    "250810": {"name": "Bentonite",                         "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    "252100": {"name": "Limestone flux",                    "chapter": "25", "sector": "minerals",      "algeria_resource": True},
    # ── Textiles & Leather ──
    "520100": {"name": "Cotton not carded combed",          "chapter": "52", "sector": "textiles",      "algeria_resource": True},
    "410120": {"name": "Bovine hides whole fresh",          "chapter": "41", "sector": "leather",       "algeria_resource": True},
    "510111": {"name": "Wool greasy shorn",                 "chapter": "51", "sector": "textiles",      "algeria_resource": True},
    # ── Processed Food ──
    "110100": {"name": "Wheat flour",                       "chapter": "11", "sector": "processed",     "algeria_resource": True},
    "190110": {"name": "Infant preparations cereal base",   "chapter": "19", "sector": "processed",     "algeria_resource": True},
    "200980": {"name": "Juice of other fruits vegetables",  "chapter": "20", "sector": "processed",     "algeria_resource": True},
    "200560": {"name": "Asparagus prepared preserved",      "chapter": "20", "sector": "processed",     "algeria_resource": False},
    "200210": {"name": "Tomatoes whole or pieces canned",   "chapter": "20", "sector": "processed",     "algeria_resource": True},
    "160414": {"name": "Tunas skipjack prepared",           "chapter": "16", "sector": "processed",     "algeria_resource": True},
    # ── Chemicals & Fertilizers ──
    "310210": {"name": "Urea whether or not in solution",   "chapter": "31", "sector": "chemicals",     "algeria_resource": True},
    "310230": {"name": "Ammonium nitrate",                  "chapter": "31", "sector": "chemicals",     "algeria_resource": True},
    "280110": {"name": "Chlorine",                          "chapter": "28", "sector": "chemicals",     "algeria_resource": True},
    "290110": {"name": "Acyclic hydrocarbons saturated",    "chapter": "29", "sector": "chemicals",     "algeria_resource": True},
}

# FAO item codes → HS code mapping
FAO_ITEMS = {
    "080410": {"fao_code": "567",  "fao_name": "Dates"},
    "080290": {"fao_code": "221",  "fao_name": "Pistachios, with shell"},
    "080610": {"fao_code": "560",  "fao_name": "Grapes"},
    "080510": {"fao_code": "490",  "fao_name": "Oranges"},
    "080520": {"fao_code": "495",  "fao_name": "Tangerines, mandarins, clem."},
    "080550": {"fao_code": "497",  "fao_name": "Lemons and limes"},
    "070200": {"fao_code": "388",  "fao_name": "Tomatoes"},
    "070700": {"fao_code": "397",  "fao_name": "Cucumbers and gherkins"},
    "070930": {"fao_code": "399",  "fao_name": "Eggplants (aubergines)"},
    "070190": {"fao_code": "116",  "fao_name": "Potatoes"},
    "070310": {"fao_code": "378",  "fao_name": "Onions and shallots, dry"},
    "100190": {"fao_code": "15",   "fao_name": "Wheat"},
    "100300": {"fao_code": "44",   "fao_name": "Barley"},
    "100590": {"fao_code": "56",   "fao_name": "Maize (corn)"},
    "150910": {"fao_code": "260",  "fao_name": "Olives"},
    "120600": {"fao_code": "267",  "fao_name": "Sunflower seed"},
    "020410": {"fao_code": "977",  "fao_name": "Meat of sheep, fresh"},
    "040110": {"fao_code": "882",  "fao_name": "Milk, whole fresh cow"},
    "030269": {"fao_code": "1750", "fao_name": "Fish, body oil"},
    "270900": {"fao_code": "2714", "fao_name": "Crude petroleum"},
    "520100": {"fao_code": "328",  "fao_name": "Seed cotton"},
    "510111": {"fao_code": "987",  "fao_name": "Wool, greasy"},
    "310210": {"fao_code": "3102", "fao_name": "Urea"},
}

ALGERIA_FAO_CODE = "4"          # Algeria's FAO area code
ALGERIA_COMTRADE_CODE = "12"    # Algeria's UN Comtrade reporter code

# ─── Rate-Limited Requester ────────────────────────────────────────────────────

class APIClient:
    """
    Smart API client with:
    - Per-domain rate limiting
    - Disk cache (never re-fetch same URL+params)
    - Exponential backoff on 429
    - Progress resumption
    """
    def __init__(self):
        self.call_counts = {}   # domain → count this hour
        self.last_call = {}     # domain → timestamp of last call

    def _domain(self, url):
        return url.split("/")[2]

    def get(self, url, params=None, delay=1.5, cache_key=None, headers=None):
        if cache_key:
            cache_path = Path(f"data/cache/{cache_key}.json")
            if cache_path.exists():
                with open(cache_path) as f:
                    return json.load(f)

        domain = self._domain(url)

        # Enforce minimum delay per domain
        if domain in self.last_call:
            elapsed = time.time() - self.last_call[domain]
            if elapsed < delay:
                time.sleep(delay - elapsed)

        for attempt in range(5):
            try:
                r = requests.get(url, params=params, headers=headers or {}, timeout=45)

                self.last_call[domain] = time.time()

                if r.status_code == 429:
                    wait = 65 * (attempt + 1)
                    log.warning(f"    ⏳ Rate limited on {domain}. Waiting {wait}s (attempt {attempt+1}/5)...")
                    time.sleep(wait)
                    continue

                if r.status_code == 500:
                    log.warning(f"    ⚠ Server error 500 on {url} — skipping")
                    return None

                r.raise_for_status()
                data = r.json()

                if cache_key:
                    with open(f"data/cache/{cache_key}.json", "w") as f:
                        json.dump(data, f)

                return data

            except requests.exceptions.Timeout:
                log.warning(f"    ⏱ Timeout attempt {attempt+1}/5 — waiting 10s")
                time.sleep(10)
            except requests.exceptions.ConnectionError:
                log.warning(f"    🔌 Connection error attempt {attempt+1}/5 — waiting 20s")
                time.sleep(20)
            except Exception as e:
                log.error(f"    ❌ Unexpected error: {e}")
                time.sleep(5)

        log.error(f"    💀 All attempts failed for {url}")
        return None

client = APIClient()

# ─── TABLE 1: UN Comtrade — World Imports (ALL countries, ALL years, partner breakdown) ────
# This is the BIG table. Strategy:
#   - Loop: each HS code × each year × paginate if needed
#   - partner = "" (all partners) so we get bilateral flows: importer ← exporter
#   - All fields from API kept as-is + computed unit_value_usd_per_kg

def fetch_comtrade_world_imports():
    log.info("=" * 70)
    log.info("TABLE 1: UN Comtrade — All world imports, all years, bilateral")
    log.info(f"  Products: {len(HS_PRODUCTS)}  |  Years: {YEARS}")
    log.info("  (Each row = Country A imports HS_CODE from Country B in Year Y)")
    log.info("=" * 70)

    BASE = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    all_records = []
    total_calls = len(HS_PRODUCTS) * len(YEARS)
    call_num = 0

    for hs, hs_meta in HS_PRODUCTS.items():
        hs4 = hs[:4]
        for year in YEARS:
            call_num += 1
            progress_key = f"comtrade_world_{hs4}_{year}"

            log.info(f"  [{call_num}/{total_calls}] HS {hs4} | {year} | {hs_meta['name'][:40]}")

            params = {
                "reporterCode": "",      # ALL importing countries
                "period": year,
                "partnerCode": "",       # ALL exporting partners (bilateral)
                "cmdCode": hs4,
                "flowCode": "M",         # M = Import
                "maxRecords": "500",
                "format": "JSON",
                "breakdownMode": "classic",
                "includeDesc": "true",
            }

            data = client.get(BASE, params=params, delay=1.5, cache_key=progress_key)

            if data and "data" in data and data["data"]:
                rows = data["data"]
                log.info(f"    → {len(rows)} bilateral trade rows")

                for rec in rows:
                    val = float(rec.get("primaryValue") or 0)
                    wgt = float(rec.get("netWgt") or 0)
                    qty = float(rec.get("qty") or 0)

                    all_records.append({
                        # ── HS / Product ──
                        "hs_code_4digit":           hs4,
                        "hs_code_6digit":           hs,
                        "product_name":             hs_meta["name"],
                        "product_sector":           hs_meta["sector"],
                        "product_chapter":          hs_meta["chapter"],
                        "algeria_has_resource":     hs_meta["algeria_resource"],
                        # ── Importer (Reporter) ──
                        "importer_code":            str(rec.get("reporterCode", "")),
                        "importer_name":            rec.get("reporterDesc", ""),
                        "importer_iso3":            rec.get("reporterISO", ""),
                        # ── Exporter (Partner) ──
                        "exporter_code":            str(rec.get("partnerCode", "")),
                        "exporter_name":            rec.get("partnerDesc", ""),
                        "exporter_iso3":            rec.get("partnerISO", ""),
                        # ── Time ──
                        "year":                     int(rec.get("period", year)),
                        # ── Trade Values ──
                        "trade_value_usd":          val,
                        "net_weight_kg":            wgt,
                        "quantity_alt":             qty,
                        "quantity_unit":            rec.get("qtyUnitAbbr", ""),
                        # ── Derived / Computed ──
                        "unit_value_usd_per_kg":    round(val / wgt, 4) if wgt > 0 else None,
                        # ── API Metadata ──
                        "flow_code":                rec.get("flowCode", "M"),
                        "flow_desc":                rec.get("flowDesc", "Import"),
                        "classification":           rec.get("classification", "HS"),
                        "is_leaf_code":             rec.get("isLeaf", ""),
                        "customs_proc_code":        rec.get("customsCode", ""),
                        "mode_of_transport_code":   rec.get("motCode", ""),
                        "aggregate_level":          rec.get("aggrlevel", ""),
                        "data_source":              "UN_Comtrade_preview_API",
                    })
            else:
                log.warning(f"    → No data returned")

    df = pd.DataFrame(all_records)
    if not df.empty:
        df.to_csv("data/raw/01_comtrade_world_imports.csv", index=False)
        log.info(f"\n  ✓ TABLE 1 complete: {len(df):,} rows, {df['importer_name'].nunique()} countries, {df['hs_code_6digit'].nunique()} products")
    else:
        log.warning("  ⚠ TABLE 1: No data collected")
    return df


# ─── TABLE 2: FAO — Algeria Production, Area, Yield (all crops/livestock/fish) ───

def fetch_fao_algeria_production():
    log.info("=" * 70)
    log.info("TABLE 2: FAO FAOSTAT — Algeria full production data")
    log.info("  Elements: Production (tonnes) + Area harvested (ha) + Yield")
    log.info("=" * 70)

    BASE = "https://fenixservices.fao.org/faostat/api/v1/en/data"
    fao_item_codes = ",".join([v["fao_code"] for v in FAO_ITEMS.values()])
    all_records = []

    datasets = [
        ("QCL", "Crops and livestock products", ["5510", "5312", "5419"]),
        # 5510=Production, 5312=Area harvested, 5419=Yield
    ]

    for dataset_code, dataset_name, elements in datasets:
        for element_code in elements:
            cache_key = f"fao_{dataset_code}_{element_code}_algeria"
            log.info(f"  Fetching {dataset_name} | element {element_code}")

            params = {
                "area":        ALGERIA_FAO_CODE,
                "element":     element_code,
                "item":        fao_item_codes,
                "year":        ",".join(["2015","2016","2017","2018","2019","2020","2021","2022"]),
                "output_type": "json",
            }

            data = client.get(f"{BASE}/{dataset_code}", params=params,
                              delay=1.0, cache_key=cache_key)

            if data and "data" in data:
                element_label = {
                    "5510": "production_tonnes",
                    "5312": "area_harvested_ha",
                    "5419": "yield_kg_per_ha",
                }.get(element_code, f"element_{element_code}")

                for rec in data["data"]:
                    fao_item_code = str(rec.get("Item Code", ""))
                    hs_code = next(
                        (hs for hs, v in FAO_ITEMS.items() if v["fao_code"] == fao_item_code),
                        None
                    )
                    all_records.append({
                        "hs_code_6digit":         hs_code,
                        "fao_item_code":          fao_item_code,
                        "fao_item_name":          rec.get("Item", ""),
                        "fao_item_group":         rec.get("Item Group", ""),
                        "fao_area_code":          rec.get("Area Code", ""),
                        "fao_area_name":          rec.get("Area", ""),
                        "fao_element_code":       element_code,
                        "fao_element_name":       rec.get("Element", ""),
                        "metric_column":          element_label,
                        "value":                  float(rec.get("Value") or 0),
                        "unit":                   rec.get("Unit", ""),
                        "year":                   int(rec.get("Year", 0)),
                        "data_flag":              rec.get("Flag", ""),
                        "flag_description":       rec.get("Flag Description", ""),
                        "note":                   rec.get("Note", ""),
                        "data_source":            "FAO_FAOSTAT_QCL",
                    })
                log.info(f"    → {len([r for r in all_records if r['fao_element_code']==element_code])} records")

    df = pd.DataFrame(all_records)
    if not df.empty:
        df.to_csv("data/raw/02_algeria_resources_fao.csv", index=False)
        log.info(f"\n  ✓ TABLE 2 complete: {len(df):,} rows")
    return df


# ─── TABLE 3: FAO — Inputs (fertilizer, pesticide, machinery use in Algeria) ───

def fetch_fao_algeria_inputs():
    log.info("=" * 70)
    log.info("TABLE 3: FAO — Agricultural inputs Algeria (fertilizers, pesticides)")
    log.info("=" * 70)

    BASE = "https://fenixservices.fao.org/faostat/api/v1/en/data/RI"
    cache_key = "fao_inputs_algeria"
    all_records = []

    params = {
        "area":        ALGERIA_FAO_CODE,
        "element":     "5159,5161,5163",   # Use, Import quantity, Import value
        "item":        "1357,1358,1359,1360,1361,1362",  # Fertilizer types
        "year":        "2015,2016,2017,2018,2019,2020,2021",
        "output_type": "json",
    }

    data = client.get(BASE, params=params, delay=1.0, cache_key=cache_key)
    if data and "data" in data:
        for rec in data["data"]:
            all_records.append({
                "fao_area":        rec.get("Area", "Algeria"),
                "fao_item_code":   rec.get("Item Code", ""),
                "fao_item_name":   rec.get("Item", ""),
                "fao_element":     rec.get("Element", ""),
                "value":           float(rec.get("Value") or 0),
                "unit":            rec.get("Unit", ""),
                "year":            int(rec.get("Year", 0)),
                "data_flag":       rec.get("Flag", ""),
                "data_source":     "FAO_FAOSTAT_ResourcesInputs",
            })
        log.info(f"  ✓ TABLE 3: {len(all_records)} input records")

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/03_algeria_resources_fao_inputs.csv", index=False)
    return df


# ─── TABLE 4: FAO — Land Use & Irrigation ─────────────────────────────────────

def fetch_fao_algeria_land():
    log.info("=" * 70)
    log.info("TABLE 4: FAO — Land use & irrigation Algeria")
    log.info("=" * 70)

    BASE = "https://fenixservices.fao.org/faostat/api/v1/en/data/RL"
    cache_key = "fao_land_algeria"
    all_records = []

    params = {
        "area":        ALGERIA_FAO_CODE,
        "element":     "5110",    # Area
        "item":        "6601,6602,6620,6655,6656,6657,6659,6661,6703",
        "year":        "2015,2016,2017,2018,2019,2020,2021",
        "output_type": "json",
    }

    data = client.get(BASE, params=params, delay=1.0, cache_key=cache_key)
    if data and "data" in data:
        for rec in data["data"]:
            all_records.append({
                "fao_area":            "Algeria",
                "land_type_code":      rec.get("Item Code", ""),
                "land_type_name":      rec.get("Item", ""),
                "element":             rec.get("Element", ""),
                "area_1000_ha":        float(rec.get("Value") or 0),
                "unit":                rec.get("Unit", ""),
                "year":                int(rec.get("Year", 0)),
                "data_flag":           rec.get("Flag", ""),
                "data_source":         "FAO_FAOSTAT_LandUse",
            })
        log.info(f"  ✓ TABLE 4: {len(all_records)} land use records")

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/04_algeria_resources_fao_land.csv", index=False)
    return df


# ─── TABLE 5: FAO — Algeria Domestic Producer Prices ─────────────────────────

def fetch_fao_algeria_prices():
    log.info("=" * 70)
    log.info("TABLE 5: FAO — Algeria domestic producer prices")
    log.info("=" * 70)

    BASE = "https://fenixservices.fao.org/faostat/api/v1/en/data/PP"
    fao_item_codes = ",".join([v["fao_code"] for v in FAO_ITEMS.values()])
    cache_key = "fao_producer_prices_algeria"
    all_records = []

    params = {
        "area":        ALGERIA_FAO_CODE,
        "element":     "5531",    # Producer price (USD/tonne)
        "item":        fao_item_codes,
        "year":        "2015,2016,2017,2018,2019,2020,2021,2022",
        "output_type": "json",
    }

    data = client.get(BASE, params=params, delay=1.0, cache_key=cache_key)
    if data and "data" in data:
        for rec in data["data"]:
            fao_item_code = str(rec.get("Item Code", ""))
            hs_code = next(
                (hs for hs, v in FAO_ITEMS.items() if v["fao_code"] == fao_item_code),
                None
            )
            all_records.append({
                "hs_code_6digit":           hs_code,
                "fao_item_code":            fao_item_code,
                "fao_item_name":            rec.get("Item", ""),
                "fao_area":                 "Algeria",
                "price_usd_per_tonne":      float(rec.get("Value") or 0),
                "unit":                     rec.get("Unit", ""),
                "year":                     int(rec.get("Year", 0)),
                "data_flag":                rec.get("Flag", ""),
                "flag_description":         rec.get("Flag Description", ""),
                "data_source":              "FAO_FAOSTAT_ProducerPrices",
            })
        log.info(f"  ✓ TABLE 5: {len(all_records)} price records")

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/05_algeria_resources_fao_prices.csv", index=False)
    return df


# ─── TABLE 6: World Bank — Algeria Resource Rents & Reserves ─────────────────

def fetch_worldbank_algeria_resources():
    log.info("=" * 70)
    log.info("TABLE 6: World Bank — Algeria resource rents and natural capital")
    log.info("=" * 70)

    BASE = "https://api.worldbank.org/v2/country/DZA/indicator"
    all_records = []

    # All resource-related World Bank indicators
    indicators = {
        "NY.GDP.TOTL.RT.ZS":   "total_natural_resource_rents_pct_gdp",
        "NY.GDP.PETR.RT.ZS":   "oil_rents_pct_gdp",
        "NY.GDP.NGAS.RT.ZS":   "natural_gas_rents_pct_gdp",
        "NY.GDP.COAL.RT.ZS":   "coal_rents_pct_gdp",
        "NY.GDP.MINR.RT.ZS":   "mineral_rents_pct_gdp",
        "NY.GDP.FRST.RT.ZS":   "forest_rents_pct_gdp",
        "EP.PMP.SGAS.CD":      "pump_price_gasoline_usd_per_liter",
        "EP.PMP.DESL.CD":      "pump_price_diesel_usd_per_liter",
        "EG.ELC.PROD.KH":      "electricity_production_kwh",
        "EG.ELC.RNEW.ZS":      "renewable_electricity_pct",
        "EG.USE.PCAP.KG.OE":   "energy_use_per_capita_kg_oil_equiv",
        "EG.FEC.RNEW.ZS":      "renewable_energy_consumption_pct",
        "AG.LND.TOTL.K2":      "land_area_sq_km",
        "AG.LND.AGRI.ZS":      "agricultural_land_pct",
        "AG.LND.ARBL.ZS":      "arable_land_pct",
        "AG.LND.IRIG.AG.ZS":   "irrigated_land_pct_cropland",
        "AG.LND.FRST.ZS":      "forest_area_pct",
        "AG.YLD.CREL.KG":      "cereal_yield_kg_per_ha",
        "ER.FSH.AQUA.MT":      "aquaculture_production_mt",
        "ER.FSH.CAPT.MT":      "capture_fisheries_production_mt",
        "NV.AGR.TOTL.ZS":      "agriculture_value_added_pct_gdp",
        "NV.MNF.TOTL.ZS.UN":   "manufacturing_value_added_pct_gdp",
        "TX.VAL.FUEL.ZS.UN":   "fuel_exports_pct_merchandise_exports",
        "TX.VAL.MMTL.ZS.UN":   "ores_metals_exports_pct_merchandise",
        "TX.VAL.AGRI.ZS.UN":   "food_exports_pct_merchandise",
        "TM.VAL.FUEL.ZS.UN":   "fuel_imports_pct_merchandise_imports",
        "TM.VAL.MMTL.ZS.UN":   "ores_metals_imports_pct_merchandise",
        "TM.VAL.AGRI.ZS.UN":   "food_imports_pct_merchandise",
    }

    for wb_indicator, col_name in indicators.items():
        cache_key = f"wb_dza_{wb_indicator}"
        url = f"{BASE}/{wb_indicator}"
        params = {
            "format":   "json",
            "date":     "2010:2022",
            "per_page": "50",
        }

        data = client.get(url, params=params, delay=0.5, cache_key=cache_key)

        if data and len(data) > 1 and data[1]:
            for entry in data[1]:
                all_records.append({
                    "country_code":     "DZA",
                    "country_name":     "Algeria",
                    "wb_indicator":     wb_indicator,
                    "indicator_label":  col_name,
                    "indicator_name":   entry.get("indicator", {}).get("value", ""),
                    "year":             int(entry.get("date", 0)),
                    "value":            entry.get("value"),
                    "decimal":          entry.get("decimal", ""),
                    "data_source":      "WorldBank_API",
                })

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/06_algeria_resources_worldbank.csv", index=False)
    log.info(f"  ✓ TABLE 6: {len(df):,} World Bank resource records")
    return df


# ─── TABLE 7: FAO — World Commodity Prices Monthly ───────────────────────────

def fetch_fao_world_prices_monthly():
    log.info("=" * 70)
    log.info("TABLE 7: FAO — World commodity prices monthly")
    log.info("=" * 70)

    BASE = "https://fenixservices.fao.org/faostat/api/v1/en/data/BCFP"
    cache_key = "fao_world_commodity_prices_monthly"
    all_records = []

    params = {
        "area":        "5000",    # World
        "element":     "5532",    # Monthly price USD
        "item":        ",".join([v["fao_code"] for v in FAO_ITEMS.values()]),
        "year":        "2018,2019,2020,2021,2022,2023",
        "output_type": "json",
    }

    data = client.get(BASE, params=params, delay=1.0, cache_key=cache_key)

    if data and "data" in data:
        for rec in data["data"]:
            fao_item_code = str(rec.get("Item Code", ""))
            hs_code = next(
                (hs for hs, v in FAO_ITEMS.items() if v["fao_code"] == fao_item_code),
                None
            )
            all_records.append({
                "hs_code_6digit":       hs_code,
                "fao_item_code":        fao_item_code,
                "fao_item_name":        rec.get("Item", ""),
                "year":                 int(rec.get("Year", 0)),
                "month_code":           rec.get("Months Code", ""),
                "month_name":           rec.get("Months", ""),
                "price_usd":            float(rec.get("Value") or 0),
                "unit":                 rec.get("Unit", ""),
                "data_flag":            rec.get("Flag", ""),
                "data_source":          "FAO_FAOSTAT_CommodityPricesMonthly",
            })
        log.info(f"  ✓ TABLE 7: {len(all_records)} monthly price records")
    else:
        log.warning("  TABLE 7: Monthly prices not returned — trying annual fallback")

        BASE2 = "https://fenixservices.fao.org/faostat/api/v1/en/data/PP"
        params2 = {
            "area":        "5000",
            "element":     "5532",
            "item":        ",".join([v["fao_code"] for v in FAO_ITEMS.values()]),
            "year":        "2018,2019,2020,2021,2022",
            "output_type": "json",
        }
        data2 = client.get(BASE2, params=params2, delay=1.0, cache_key="fao_world_prices_annual")
        if data2 and "data" in data2:
            for rec in data2["data"]:
                fao_item_code = str(rec.get("Item Code", ""))
                hs_code = next(
                    (hs for hs, v in FAO_ITEMS.items() if v["fao_code"] == fao_item_code),
                    None
                )
                all_records.append({
                    "hs_code_6digit":   hs_code,
                    "fao_item_code":    fao_item_code,
                    "fao_item_name":    rec.get("Item", ""),
                    "year":             int(rec.get("Year", 0)),
                    "month_code":       "annual",
                    "month_name":       "Annual",
                    "price_usd":        float(rec.get("Value") or 0),
                    "unit":             rec.get("Unit", ""),
                    "data_flag":        rec.get("Flag", ""),
                    "data_source":      "FAO_FAOSTAT_ProducerPrices_WorldAnnual",
                })

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/07_fao_commodity_prices_monthly.csv", index=False)
    return df


# ─── TABLE 8: World Bank — Commodity Price Index ──────────────────────────────

def fetch_worldbank_commodity_prices():
    log.info("=" * 70)
    log.info("TABLE 8: World Bank — commodity price indices")
    log.info("=" * 70)

    # WB Pink Sheet commodity prices API
    BASE = "https://api.worldbank.org/v2/indicator"
    all_records = []

    commodity_indicators = {
        "PBARLE":   {"name": "Barley price USD per mt",              "hs": "100300"},
        "PMAIZMT":  {"name": "Maize price USD per mt",               "hs": "100590"},
        "PWHEAMT":  {"name": "Wheat price USD per mt",               "hs": "100190"},
        "PRICENPQ": {"name": "Rice price USD per mt",                "hs": "100630"},
        "POILWTI":  {"name": "Crude oil WTI USD per bbl",            "hs": "270900"},
        "PNGASEU":  {"name": "Natural gas Europe USD per mmbtu",     "hs": "271111"},
        "PCOALAU":  {"name": "Coal Australian USD per mt",           "hs": "270119"},
        "PSOYB":    {"name": "Soybeans price USD per mt",            "hs": "120100"},
        "PSUNFL":   {"name": "Sunflower oil price USD per mt",       "hs": "120600"},
        "POLVOILEX": {"name": "Olive oil price USD per mt",          "hs": "150910"},
        "PCOTTIND": {"name": "Cotton index USD per kg",              "hs": "520100"},
        "PPHOSPH":  {"name": "Phosphate rock USD per mt",            "hs": "251010"},
        "PUREA":    {"name": "Urea price USD per mt",                "hs": "310210"},
        "PFISH":    {"name": "Fish meal price USD per mt",           "hs": "030269"},
    }

    for wb_code, meta in commodity_indicators.items():
        cache_key = f"wb_commodity_{wb_code}"
        url = f"{BASE}/CMO/{wb_code}"
        params = {
            "format":   "json",
            "date":     "2015:2023",
            "per_page": "100",
        }

        data = client.get(url, params=params, delay=0.5, cache_key=cache_key)

        if data and len(data) > 1 and data[1]:
            for entry in data[1]:
                all_records.append({
                    "wb_commodity_code":    wb_code,
                    "commodity_name":       meta["name"],
                    "hs_code_6digit":       meta["hs"],
                    "year":                 int(entry.get("date", 0)),
                    "price_value":          entry.get("value"),
                    "indicator_id":         entry.get("indicator", {}).get("id", ""),
                    "indicator_label":      entry.get("indicator", {}).get("value", ""),
                    "data_source":          "WorldBank_CommodityPrices_PinkSheet",
                })

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/08_worldbank_commodity_index.csv", index=False)
    log.info(f"  ✓ TABLE 8: {len(df):,} commodity price index records")
    return df


# ─── TABLE 9: Comtrade — Unit Values (implicit price = value/weight) ─────────
# Already computed in TABLE 1 (unit_value_usd_per_kg column)
# This table aggregates it cleanly: median price per HS per year per trade lane

def build_unit_values_table(df_imports):
    log.info("=" * 70)
    log.info("TABLE 9: Derived unit values (implicit price per kg from trade flows)")
    log.info("=" * 70)

    if df_imports is None or df_imports.empty:
        log.warning("  TABLE 9: No import data to derive from")
        pd.DataFrame().to_csv("data/raw/09_comtrade_unit_values.csv", index=False)
        return pd.DataFrame()

    df = df_imports[df_imports["unit_value_usd_per_kg"].notna()].copy()
    df = df[df["unit_value_usd_per_kg"] > 0]
    df = df[df["net_weight_kg"] > 1000]  # filter noise (tiny trades)

    df_uv = df[[
        "hs_code_6digit", "product_name", "product_sector",
        "importer_name", "importer_iso3",
        "exporter_name", "exporter_iso3",
        "year",
        "trade_value_usd", "net_weight_kg",
        "unit_value_usd_per_kg",
    ]].copy()

    df_uv.to_csv("data/raw/09_comtrade_unit_values.csv", index=False)
    log.info(f"  ✓ TABLE 9: {len(df_uv):,} unit value records")
    return df_uv


# ─── TABLE 10: Comtrade — What Algeria Imports (resource gaps) ───────────────

def fetch_algeria_imports():
    log.info("=" * 70)
    log.info("TABLE 10: UN Comtrade — What Algeria imports (= resource gaps)")
    log.info("  (Algeria importing a product = it lacks domestic supply)")
    log.info("=" * 70)

    BASE = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    all_records = []
    total_calls = len(HS_PRODUCTS) * len(YEARS)
    call_num = 0

    for hs, hs_meta in HS_PRODUCTS.items():
        hs4 = hs[:4]
        for year in YEARS:
            call_num += 1
            cache_key = f"comtrade_algeria_import_{hs4}_{year}"
            log.info(f"  [{call_num}/{total_calls}] Algeria imports HS {hs4} | {year}")

            params = {
                "reporterCode": ALGERIA_COMTRADE_CODE,
                "period":       year,
                "partnerCode":  "",       # all origins
                "cmdCode":      hs4,
                "flowCode":     "M",
                "maxRecords":   "500",
                "format":       "JSON",
                "breakdownMode":"classic",
                "includeDesc":  "true",
            }

            data = client.get(BASE, params=params, delay=1.5, cache_key=cache_key)

            if data and "data" in data and data["data"]:
                for rec in data["data"]:
                    val = float(rec.get("primaryValue") or 0)
                    wgt = float(rec.get("netWgt") or 0)
                    all_records.append({
                        "hs_code_4digit":           hs4,
                        "hs_code_6digit":           hs,
                        "product_name":             hs_meta["name"],
                        "product_sector":           hs_meta["sector"],
                        "algeria_claims_resource":  hs_meta["algeria_resource"],
                        "importer_code":            ALGERIA_COMTRADE_CODE,
                        "importer_name":            "Algeria",
                        "exporter_code":            str(rec.get("partnerCode", "")),
                        "exporter_name":            rec.get("partnerDesc", ""),
                        "exporter_iso3":            rec.get("partnerISO", ""),
                        "year":                     int(rec.get("period", year)),
                        "import_value_usd":         val,
                        "net_weight_kg":            wgt,
                        "quantity_alt":             float(rec.get("qty") or 0),
                        "quantity_unit":            rec.get("qtyUnitAbbr", ""),
                        "unit_value_usd_per_kg":    round(val / wgt, 4) if wgt > 0 else None,
                        "flow_code":                "M",
                        "classification":           rec.get("classification", "HS"),
                        "data_source":              "UN_Comtrade_preview_API",
                        "interpretation":           "algeria_imports_this_product",
                    })

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/10_algeria_imports_comtrade.csv", index=False)
    log.info(f"  ✓ TABLE 10: {len(df):,} Algeria import records")
    return df


# ─── TABLE 11: World Bank — Full country metadata for ALL trade countries ─────

def fetch_all_country_metadata():
    log.info("=" * 70)
    log.info("TABLE 11: World Bank — metadata all countries (GDP, pop, trade)")
    log.info("=" * 70)

    BASE = "https://api.worldbank.org/v2/country/all/indicator"
    all_records = []

    indicators = {
        "NY.GDP.MKTP.CD":       "gdp_current_usd",
        "NY.GDP.PCAP.CD":       "gdp_per_capita_usd",
        "NY.GDP.MKTP.KD.ZG":    "gdp_growth_pct",
        "SP.POP.TOTL":          "population_total",
        "TM.VAL.MRCH.CD.WT":    "total_merchandise_imports_usd",
        "TX.VAL.MRCH.CD.WT":    "total_merchandise_exports_usd",
        "NE.TRD.GNFS.ZS":       "trade_pct_gdp",
        "BX.KLT.DINV.CD.WD":    "fdi_net_inflows_usd",
        "IC.LGL.DURS":          "time_to_enforce_contract_days",
        "SL.UEM.TOTL.ZS":       "unemployment_pct",
    }

    for wb_ind, col_name in indicators.items():
        cache_key = f"wb_all_countries_{wb_ind}_2022"
        url = f"{BASE}/{wb_ind}"
        params = {
            "format":   "json",
            "date":     "2022",
            "per_page": "300",
        }

        data = client.get(url, params=params, delay=0.5, cache_key=cache_key)

        if data and len(data) > 1 and data[1]:
            for entry in data[1]:
                all_records.append({
                    "country_iso3":     entry.get("countryiso3code", ""),
                    "country_code":     entry.get("country", {}).get("id", ""),
                    "country_name":     entry.get("country", {}).get("value", ""),
                    "year":             int(entry.get("date", 0)),
                    "wb_indicator":     wb_ind,
                    "indicator_label":  col_name,
                    "value":            entry.get("value"),
                    "data_source":      "WorldBank_API",
                })

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/11_country_metadata.csv", index=False)
    log.info(f"  ✓ TABLE 11: {len(df):,} country metadata records")
    return df


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    start = datetime.now()
    log.info("╔══════════════════════════════════════════════════════════════╗")
    log.info("║   ALGERIA TRADE & RESOURCES — FULL DATA COLLECTION          ║")
    log.info(f"║   Started: {start.strftime('%Y-%m-%d %H:%M:%S')}                          ║")
    log.info("╚══════════════════════════════════════════════════════════════╝")
    log.info(f"Products (HS codes): {len(HS_PRODUCTS)}")
    log.info(f"Years: {YEARS}")
    log.info(f"Cached calls will be SKIPPED automatically\n")

    # Run all collectors
    df_imports   = fetch_comtrade_world_imports()      # TABLE 1 — big one
    df_prod      = fetch_fao_algeria_production()      # TABLE 2
    df_inputs    = fetch_fao_algeria_inputs()          # TABLE 3
    df_land      = fetch_fao_algeria_land()            # TABLE 4
    df_dza_price = fetch_fao_algeria_prices()          # TABLE 5
    df_wb_res    = fetch_worldbank_algeria_resources() # TABLE 6
    df_fao_price = fetch_fao_world_prices_monthly()    # TABLE 7
    df_wb_comm   = fetch_worldbank_commodity_prices()  # TABLE 8
    df_uv        = build_unit_values_table(df_imports) # TABLE 9 (derived)
    df_dza_imp   = fetch_algeria_imports()             # TABLE 10
    df_meta      = fetch_all_country_metadata()        # TABLE 11

    # Summary
    elapsed = (datetime.now() - start).seconds // 60
    log.info("\n╔══════════════════════════════════════════════════════════════╗")
    log.info("║   COLLECTION COMPLETE                                        ║")
    log.info("╠══════════════════════════════════════════════════════════════╣")
    tables = [
        ("01_comtrade_world_imports.csv",       df_imports,   "Global imports ALL countries bilateral"),
        ("02_algeria_resources_fao.csv",         df_prod,      "Algeria production + area + yield"),
        ("03_algeria_resources_fao_inputs.csv",  df_inputs,    "Algeria fertilizers & inputs"),
        ("04_algeria_resources_fao_land.csv",    df_land,      "Algeria land use & irrigation"),
        ("05_algeria_resources_fao_prices.csv",  df_dza_price, "Algeria domestic producer prices"),
        ("06_algeria_resources_worldbank.csv",   df_wb_res,    "Algeria resource rents (World Bank)"),
        ("07_fao_commodity_prices_monthly.csv",  df_fao_price, "World commodity prices monthly"),
        ("08_worldbank_commodity_index.csv",     df_wb_comm,   "World Bank commodity price index"),
        ("09_comtrade_unit_values.csv",          df_uv,        "Implicit prices (value/kg)"),
        ("10_algeria_imports_comtrade.csv",      df_dza_imp,   "Algeria imports = resource gaps"),
        ("11_country_metadata.csv",              df_meta,      "All countries GDP pop trade"),
    ]
    for fname, df, desc in tables:
        rows = len(df) if df is not None and not df.empty else 0
        log.info(f"║  {fname:<42} {rows:>7,} rows  ║")
    log.info(f"╠══════════════════════════════════════════════════════════════╣")
    log.info(f"║  Total time: ~{elapsed} minutes                                    ║")
    log.info(f"║  All files saved in: data/raw/                               ║")
    log.info("╚══════════════════════════════════════════════════════════════╝")


if __name__ == "__main__":
    main()

---
## 🔧 FAO Bulk CSV Patch (Tables 2–5 and 7)
The FAO API (fenixservices) sometimes returns 521 errors.
This patch downloads the full bulk CSV files directly — more reliable, no rate limits.

Run this cell if Tables 2–5 are empty after the main pipeline above.


In [ ]:
"""
=============================================================================
PATCH: Fetch FAO + World Bank data using reliable methods
=============================================================================
FAO API (fenixservices) is DOWN with 521 errors.
Fix: Use FAO bulk CSV download instead — always works, no rate limits.

World Bank CMO path was wrong.
Fix: Use correct WB Pink Sheet CSV endpoint.

Run this AFTER collect_all_data.py finishes Table 1.
=============================================================================
"""

import requests
import pandas as pd
import io
import time
import zipfile
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/cache").mkdir(parents=True, exist_ok=True)

# ── FAO item codes we care about (same as main script) ──────────────────────
FAO_ITEMS = {
    "080410": {"fao_code": "567",  "name": "Dates"},
    "080290": {"fao_code": "221",  "name": "Pistachios"},
    "080610": {"fao_code": "560",  "name": "Grapes"},
    "080510": {"fao_code": "490",  "name": "Oranges"},
    "080520": {"fao_code": "495",  "name": "Tangerines"},
    "080550": {"fao_code": "497",  "name": "Lemons and limes"},
    "070200": {"fao_code": "388",  "name": "Tomatoes"},
    "070700": {"fao_code": "397",  "name": "Cucumbers"},
    "070930": {"fao_code": "399",  "name": "Eggplants"},
    "070190": {"fao_code": "116",  "name": "Potatoes"},
    "070310": {"fao_code": "378",  "name": "Onions"},
    "100190": {"fao_code": "15",   "name": "Wheat"},
    "100300": {"fao_code": "44",   "name": "Barley"},
    "100590": {"fao_code": "56",   "name": "Maize"},
    "150910": {"fao_code": "260",  "name": "Olives"},
    "120600": {"fao_code": "267",  "name": "Sunflower seed"},
    "020410": {"fao_code": "977",  "name": "Sheep meat"},
    "040110": {"fao_code": "882",  "name": "Cow milk"},
    "270900": {"fao_code": "2714", "name": "Crude petroleum"},
    "520100": {"fao_code": "328",  "name": "Seed cotton"},
    "510111": {"fao_code": "987",  "name": "Wool greasy"},
}

FAO_ITEM_CODES = set(str(v["fao_code"]) for v in FAO_ITEMS.values())
HS_BY_FAO = {v["fao_code"]: hs for hs, v in FAO_ITEMS.items()}

ALGERIA_FAO_CODE = "4"


# ─── 1. FAO Bulk CSV Download ─────────────────────────────────────────────────
# FAO publishes full bulk CSVs at a stable URL — much more reliable than their API

def fetch_fao_via_bulk_csv():
    """
    Download FAO FAOSTAT bulk CSV files directly.
    These are stable zip files that always work even when the API is down.

    Datasets:
      QCL = Crops and Livestock Products (production, area, yield)
      PP  = Producer Prices
      RL  = Land Use
    """
    log.info("=" * 65)
    log.info("FAO Bulk CSV Download — Algeria production, prices, land")
    log.info("=" * 65)

    datasets = [
        {
            "code":     "QCL",
            "url":      "https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip",
            "filename": "Production_Crops_Livestock_E_All_Data_(Normalized).csv",
            "output":   "data/raw/02_algeria_resources_fao.csv",
            "desc":     "Production + Area + Yield",
        },
        {
            "code":     "PP",
            "url":      "https://bulks-faostat.fao.org/production/Prices_E_All_Data_(Normalized).zip",
            "filename": "Prices_E_All_Data_(Normalized).csv",
            "output":   "data/raw/05_algeria_resources_fao_prices.csv",
            "desc":     "Producer Prices",
        },
        {
            "code":     "RL",
            "url":      "https://bulks-faostat.fao.org/production/Inputs_LandUse_E_All_Data_(Normalized).zip",
            "filename": "Inputs_LandUse_E_All_Data_(Normalized).csv",
            "output":   "data/raw/04_algeria_resources_fao_land.csv",
            "desc":     "Land Use",
        },
        {
            "code":     "RI",
            "url":      "https://bulks-faostat.fao.org/production/Inputs_FertilizersNutrient_E_All_Data_(Normalized).zip",
            "filename": "Inputs_FertilizersNutrient_E_All_Data_(Normalized).csv",
            "output":   "data/raw/03_algeria_resources_fao_inputs.csv",
            "desc":     "Fertilizer Inputs",
        },
    ]

    for ds in datasets:
        cache_zip = Path(f"data/cache/fao_{ds['code']}_bulk.zip")
        log.info(f"\n  Downloading FAO {ds['code']} — {ds['desc']}")

        # Download zip if not cached
        if not cache_zip.exists():
            log.info(f"    Fetching {ds['url']}")
            try:
                r = requests.get(ds["url"], timeout=120, stream=True)
                r.raise_for_status()
                with open(cache_zip, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                log.info(f"    Downloaded: {cache_zip.stat().st_size / 1024 / 1024:.1f} MB")
            except Exception as e:
                log.error(f"    ❌ Download failed: {e}")
                continue
        else:
            log.info(f"    [CACHE] Using {cache_zip}")

        # Extract and filter for Algeria + our items
        try:
            log.info(f"    Extracting and filtering...")
            with zipfile.ZipFile(cache_zip) as z:
                # Find the right file inside the zip
                csv_name = next(
                    (n for n in z.namelist() if n.endswith(".csv") and "flag" not in n.lower()),
                    None
                )
                if not csv_name:
                    csv_name = ds["filename"]
                log.info(f"    Reading {csv_name} from zip")

                with z.open(csv_name) as f:
                    # Read in chunks — these files are large (100MB+)
                    chunks = []
                    reader = pd.read_csv(
                        f,
                        encoding="latin-1",
                        chunksize=100_000,
                        low_memory=False
                    )
                    for chunk in reader:
                        # Filter Algeria
                        if "Area Code" in chunk.columns:
                            algeria_chunk = chunk[
                                chunk["Area Code"].astype(str) == ALGERIA_FAO_CODE
                            ]
                        elif "Area Code (M49)" in chunk.columns:
                            algeria_chunk = chunk[
                                chunk["Area Code (M49)"].astype(str).str.strip("'") == "12"
                            ]
                        else:
                            algeria_chunk = chunk[
                                chunk.get("Area", pd.Series()).str.contains("Algeria", na=False)
                            ]

                        if len(algeria_chunk) > 0:
                            # For QCL, also filter by our item codes
                            if ds["code"] == "QCL" and "Item Code" in algeria_chunk.columns:
                                algeria_chunk = algeria_chunk[
                                    algeria_chunk["Item Code"].astype(str).isin(FAO_ITEM_CODES)
                                ]
                            chunks.append(algeria_chunk)

                if chunks:
                    df = pd.concat(chunks, ignore_index=True)

                    # Add HS code mapping where applicable
                    if "Item Code" in df.columns:
                        df["hs_code_6digit"] = df["Item Code"].astype(str).map(HS_BY_FAO)

                    # Add data source column
                    df["data_source"] = f"FAO_FAOSTAT_{ds['code']}_bulk_CSV"

                    df.to_csv(ds["output"], index=False)
                    log.info(f"    ✓ Saved {len(df):,} rows → {ds['output']}")
                else:
                    log.warning(f"    ⚠ No Algeria data found in {ds['code']}")

        except Exception as e:
            log.error(f"    ❌ Processing failed: {e}")
            import traceback; traceback.print_exc()

        time.sleep(1)


# ─── 2. FAO World Commodity Prices ───────────────────────────────────────────
# Use the World Food Price dataset (separate from FAOSTAT bulk)

def fetch_fao_world_prices():
    """
    FAO Food Price Index and commodity prices.
    Uses the FAO Food Price Index CSV (always available, no API needed).
    Also fetches the full bulk price CSV filtered to world averages.
    """
    log.info("\n" + "=" * 65)
    log.info("FAO World Commodity Prices — bulk CSV")
    log.info("=" * 65)

    records = []

    # FAO Food Price Index (monthly, very reliable)
    fpi_url = "https://www.fao.org/fileadmin/templates/worldfood/Reports_and_docs/food_price_indices_data_jul24.xls"
    fpi_csv_url = "https://www.fao.org/worldfoodsituation/foodpricesindex/en/"

    # Use the bulk normalized prices file we may have already downloaded
    bulk_prices_zip = Path("data/cache/fao_PP_bulk.zip")
    if bulk_prices_zip.exists():
        log.info("  Using already-downloaded PP bulk zip for world prices")
        try:
            with zipfile.ZipFile(bulk_prices_zip) as z:
                csv_name = next(n for n in z.namelist() if n.endswith(".csv") and "flag" not in n.lower())
                with z.open(csv_name) as f:
                    chunks = []
                    reader = pd.read_csv(f, encoding="latin-1", chunksize=100_000, low_memory=False)
                    for chunk in reader:
                        # World = area code 5000 or "World"
                        if "Area Code" in chunk.columns:
                            world_chunk = chunk[chunk["Area Code"].astype(str) == "5000"]
                        else:
                            world_chunk = chunk[chunk.get("Area", pd.Series()).str.contains("World", na=False)]

                        if len(world_chunk) > 0:
                            if "Item Code" in world_chunk.columns:
                                world_chunk = world_chunk[
                                    world_chunk["Item Code"].astype(str).isin(FAO_ITEM_CODES)
                                ]
                            chunks.append(world_chunk)

                if chunks:
                    df = pd.concat(chunks, ignore_index=True)
                    if "Item Code" in df.columns:
                        df["hs_code_6digit"] = df["Item Code"].astype(str).map(HS_BY_FAO)
                    df["data_source"] = "FAO_FAOSTAT_PP_WorldPrices_bulk_CSV"
                    df.to_csv("data/raw/07_fao_commodity_prices_monthly.csv", index=False)
                    log.info(f"  ✓ Saved {len(df):,} world price records")
                    return df
        except Exception as e:
            log.error(f"  Bulk approach failed: {e}")

    # Fallback: FAO Food Price Index CSV (monthly index, very lightweight)
    log.info("  Fetching FAO Food Price Index (FFPI) CSV...")
    ffpi_url = "https://www.fao.org/fileadmin/templates/worldfood/Reports_and_docs/food_price_indices_data.csv"
    try:
        r = requests.get(ffpi_url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        df["data_source"] = "FAO_FoodPriceIndex_CSV"
        df.to_csv("data/raw/07_fao_commodity_prices_monthly.csv", index=False)
        log.info(f"  ✓ Saved {len(df):,} FFPI records")
        return df
    except Exception as e:
        log.warning(f"  FFPI CSV also failed: {e}")

    log.warning("  Saving empty placeholder for TABLE 7")
    pd.DataFrame().to_csv("data/raw/07_fao_commodity_prices_monthly.csv", index=False)
    return pd.DataFrame()


# ─── 3. World Bank Commodity Prices — CORRECT endpoint ───────────────────────
# The CMO path used before was wrong.
# Correct: use the Pink Sheet Excel file directly or the WB data API with right codes

def fetch_worldbank_commodity_prices_fixed():
    """
    World Bank Pink Sheet commodity prices.
    Correct approach: download the Excel file directly from WB website,
    OR use the standard WB indicator API with correct indicator codes.
    """
    log.info("\n" + "=" * 65)
    log.info("World Bank Commodity Prices — fixed endpoint")
    log.info("=" * 65)

    all_records = []

    # CORRECT World Bank indicator codes (standard API, not CMO)
    correct_indicators = {
        "PCOMM.MAL.WHEAT":    {"name": "Wheat price USD/mt",           "hs": "100190"},
        "PCOMM.MAL.MAIZE":    {"name": "Maize price USD/mt",           "hs": "100590"},
        "PCOMM.MAL.RICE":     {"name": "Rice price USD/mt",            "hs": "100630"},
        "PCOMM.MAL.SOYBEANS": {"name": "Soybeans price USD/mt",        "hs": "120100"},
        "PCOMM.MAL.COTTON_A": {"name": "Cotton index USD/kg",          "hs": "520100"},
        "PCOMM.MAL.CRUDE_WTI":{"name": "Crude oil WTI USD/bbl",        "hs": "270900"},
        "PCOMM.MAL.NGAS_EUR": {"name": "Natural gas Europe USD/mmbtu", "hs": "271111"},
        "PCOMM.MAL.PHOSPHATE":{"name": "Phosphate rock USD/mt",        "hs": "251010"},
        "PCOMM.MAL.UREA":     {"name": "Urea price USD/mt",            "hs": "310210"},
        "PCOMM.MAL.GOLD":     {"name": "Gold USD/troy oz",             "hs": "710812"},
        "PCOMM.MAL.COPPER":   {"name": "Copper USD/mt",                "hs": "260300"},
        "PCOMM.MAL.IRON_ORE": {"name": "Iron ore USD/dmt",             "hs": "260111"},
    }

    BASE = "https://api.worldbank.org/v2/country/all/indicator"
    success = 0

    for wb_code, meta in correct_indicators.items():
        cache_key = f"data/cache/wb_comm_fixed_{wb_code.replace('.', '_')}.json"
        if Path(cache_key).exists():
            import json
            with open(cache_key) as f:
                data = json.load(f)
        else:
            url = f"{BASE}/{wb_code}"
            params = {"format": "json", "date": "2015:2023", "per_page": "200"}
            try:
                r = requests.get(url, params=params, timeout=20)
                if r.status_code == 404:
                    continue
                r.raise_for_status()
                data = r.json()
                import json
                with open(cache_key, "w") as f:
                    json.dump(data, f)
                time.sleep(0.5)
            except Exception as e:
                log.warning(f"  {wb_code}: {e}")
                continue

        if data and len(data) > 1 and data[1]:
            for entry in data[1]:
                if entry.get("value") is not None:
                    all_records.append({
                        "wb_indicator_code": wb_code,
                        "commodity_name":    meta["name"],
                        "hs_code_6digit":    meta["hs"],
                        "country":           entry.get("country", {}).get("value", ""),
                        "year":              int(entry.get("date", 0)),
                        "price_value":       entry.get("value"),
                        "data_source":       "WorldBank_commodity_API",
                    })
            success += 1

    # Fallback: Download WB Pink Sheet CSV directly
    if success == 0:
        log.info("  WB indicator API returned nothing — trying Pink Sheet direct CSV")
        pink_url = "https://thedocs.worldbank.org/en/doc/18675f1d1639c7a34d463f59263ba0a2-0050012024/related/CMO-Historical-Data-Monthly.xlsx"
        try:
            r = requests.get(pink_url, timeout=60)
            r.raise_for_status()
            # Read the Excel
            xls = pd.ExcelFile(io.BytesIO(r.content))
            log.info(f"  Pink Sheet sheets: {xls.sheet_names}")
            # Monthly prices sheet
            df_raw = pd.read_excel(io.BytesIO(r.content), sheet_name="Monthly Prices", header=4)
            df_raw["data_source"] = "WorldBank_PinkSheet_direct"
            df_raw.to_csv("data/raw/08_worldbank_commodity_index.csv", index=False)
            log.info(f"  ✓ Saved {len(df_raw):,} Pink Sheet records")
            return df_raw
        except Exception as e:
            log.error(f"  Pink Sheet also failed: {e}")

    df = pd.DataFrame(all_records)
    df.to_csv("data/raw/08_worldbank_commodity_index.csv", index=False)
    log.info(f"  ✓ Saved {len(df):,} commodity price records")
    return df


# ─── MAIN ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    log.info("Running patch for FAO + World Bank data...")
    log.info("(UN Comtrade TABLE 1 already complete — this only fixes the failed tables)\n")

    fetch_fao_via_bulk_csv()       # Tables 2, 3, 4, 5
    fetch_fao_world_prices()       # Table 7
    fetch_worldbank_commodity_prices_fixed()  # Table 8

    log.info("\n✓ Patch complete. Check data/raw/ for all files.")

    # Quick summary
    import os
    print("\n📁 Files in data/raw/:")
    for f in sorted(Path("data/raw").glob("*.csv")):
        size_kb = f.stat().st_size / 1024
        try:
            rows = sum(1 for _ in open(f)) - 1
        except:
            rows = 0
        print(f"  {f.name:<55} {rows:>8,} rows  ({size_kb:.0f} KB)")

---
## 📋 Table 12 — Algeria Export Flows
**Source:** OEC World / BACI trade database

This is the **ground truth** for what Algeria already exports and to whom.

**Two-step approach:**
1. **Fast extraction** (instant): filter rows from Table 1 where `exporter_iso3 == "DZA"` — works immediately with no extra API calls
2. **Full OEC pull** (detailed): queries the OEC BACI cube for all 5 years with pagination, auto-probing the API schema

> The OEC API schema changes periodically, so the script auto-detects the correct cube name and level names before fetching.

**Saved to:** `data/raw/12_algeria_exports_comtrade.csv`


In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/01_comtrade_world_imports.csv")
algeria_as_exporter = df[df["exporter_iso3"] == "DZA"].copy()
algeria_as_exporter.to_csv("data/raw/12_algeria_exports_comtrade.csv", index=False)
print(len(algeria_as_exporter), "rows")

In [ ]:
"""
=============================================================================
TABLE 12 — Algeria Exports via OEC BACI (auto-probing cube schema)
=============================================================================
The previous version hardcoded wrong level names. This version:
  1. Probes the cube schema first to discover the exact level names
  2. Tries multiple known name variants as fallback
  3. Paginates fully until all Algeria export rows are fetched

Output: data/raw/12_algeria_exports_comtrade.csv
=============================================================================
"""

import requests
import pandas as pd
import json
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/cache").mkdir(parents=True, exist_ok=True)

OEC_BASE  = "https://api-v2.oec.world/tesseract"
OEC_TOKEN = "6b540954e368f971"
YEARS     = [2018, 2019, 2020, 2021, 2022]
PAGE_SIZE = 5000
PROGRESS_FILE = "data/cache/progress_12_oec_v2.json"

# Candidate cube names for HS 2017 revision (covers 2017-2022)
CUBES_TO_TRY = [
    "trade_i_baci_a_17",
    "trade_i_baci_a_12",
    "trade_i_baci_a_96",
]

# Algeria's OEC country IDs to try
ALGERIA_IDS = ["afdza", "12", "dza", "DZA"]


def load_progress():
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"done_years": [], "cube": None, "exporter_level": None,
            "importer_level": None, "algeria_id": None}


def save_progress(p):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f)


def probe_cube_schema(cube):
    """Fetch cube metadata and return all available level names."""
    cache = Path(f"data/cache/oec_schema_{cube}.json")
    if cache.exists():
        with open(cache) as f:
            return json.load(f)
    try:
        time.sleep(1)
        r = requests.get(f"{OEC_BASE}/cubes/{cube}", timeout=20,
                         params={"token": OEC_TOKEN})
        if r.status_code == 200:
            data = r.json()
            with open(cache, "w") as f:
                json.dump(data, f)
            return data
    except Exception as e:
        log.warning(f"  Schema probe failed: {e}")
    return {}


def extract_levels(schema):
    """Pull all level names from cube schema."""
    levels = []
    for dim in schema.get("dimensions", []):
        for hier in dim.get("hierarchies", []):
            for lvl in hier.get("levels", []):
                levels.append(lvl["name"])
    return levels


def test_query(cube, exporter_lvl, importer_lvl, algeria_id, year=2022):
    """Test one query variant with limit=1 to check if it works."""
    params = {
        "cube":       cube,
        "drilldowns": f"HS4,{exporter_lvl},{importer_lvl},Year",
        "measures":   "Trade Value,Quantity",
        "include":    f"{exporter_lvl}:{algeria_id};Year:{year}",
        "limit":      "1,0",
        "token":      OEC_TOKEN,
    }
    try:
        time.sleep(0.8)
        r = requests.get(f"{OEC_BASE}/data.jsonrecords", params=params, timeout=20)
        if r.status_code == 200:
            data = r.json()
            return True, data
        else:
            return False, r.text[:300]
    except Exception as e:
        return False, str(e)


def discover_working_config():
    """
    Probe cubes and level name variants to find a working combination.
    Returns (cube, exporter_level, importer_level, algeria_id) or None.
    """
    log.info("🔍 Probing OEC cube schemas to find correct level names...")

    # Known level name patterns across OEC cube versions
    exporter_variants = [
        "Exporter Country", "Exporter", "Origin Country", "Origin",
        "Reporter", "Country", "exporter_country", "exporter",
    ]
    importer_variants = [
        "Importer Country", "Importer", "Destination Country", "Destination",
        "Partner", "importer_country", "importer",
    ]

    for cube in CUBES_TO_TRY:
        log.info(f"\n  Testing cube: {cube}")

        # Get schema to narrow down level names
        schema = probe_cube_schema(cube)
        if schema:
            actual_levels = extract_levels(schema)
            log.info(f"    Available levels: {actual_levels}")
            # Filter variants to only those that exist in schema
            exp_candidates = [v for v in exporter_variants if v in actual_levels] or exporter_variants
            imp_candidates = [v for v in importer_variants if v in actual_levels] or importer_variants
        else:
            exp_candidates = exporter_variants
            imp_candidates = importer_variants

        for exp_lvl in exp_candidates:
            for imp_lvl in imp_candidates:
                if exp_lvl == imp_lvl:
                    continue
                for alg_id in ALGERIA_IDS:
                    ok, result = test_query(cube, exp_lvl, imp_lvl, alg_id)
                    if ok:
                        data = result
                        n = len(data.get("data", []))
                        log.info(f"    ✅ WORKS: cube={cube} exp={exp_lvl} imp={imp_lvl} id={alg_id} → {n} rows")
                        return cube, exp_lvl, imp_lvl, alg_id
                    else:
                        # Only log first error per combo to avoid spam
                        if "Could not find" in str(result):
                            pass  # silently skip wrong level names
                        else:
                            log.info(f"    {exp_lvl}/{imp_lvl}/{alg_id}: {str(result)[:80]}")

    return None, None, None, None


def fetch_year(cube, exporter_lvl, importer_lvl, algeria_id, year):
    """Paginate through all Algeria export rows for a given year."""
    rows   = []
    offset = 0
    page   = 1

    while True:
        cache_key = f"data/cache/oec_v2_alg_{year}_p{page}.json"

        if Path(cache_key).exists():
            log.info(f"    Page {page} — CACHE")
            with open(cache_key) as f:
                data = json.load(f)
        else:
            params = {
                "cube":       cube,
                "drilldowns": f"HS4,{exporter_lvl},{importer_lvl},Year",
                "measures":   "Trade Value,Quantity",
                "include":    f"{exporter_lvl}:{algeria_id};Year:{year}",
                "limit":      f"{PAGE_SIZE},{offset}",
                "token":      OEC_TOKEN,
            }
            log.info(f"    Page {page} (offset={offset}) — fetching from OEC...")
            try:
                time.sleep(1.2)
                r = requests.get(f"{OEC_BASE}/data.jsonrecords", params=params, timeout=40)
                if r.status_code == 429:
                    log.warning("    429 — sleeping 60s")
                    time.sleep(60)
                    r = requests.get(f"{OEC_BASE}/data.jsonrecords", params=params, timeout=40)
                if r.status_code != 200:
                    log.warning(f"    HTTP {r.status_code}: {r.text[:200]}")
                    break
                data = r.json()
                with open(cache_key, "w") as f:
                    json.dump(data, f)
            except Exception as e:
                log.warning(f"    Error: {e}")
                break

        records = data.get("data", [])
        total   = data.get("page", {}).get("total", 0)
        log.info(f"    Page {page}: {len(records)} records (total={total})")

        for rec in records:
            trade_val = rec.get("Trade Value") or 0
            qty       = rec.get("Quantity") or None

            # HS4 ID from OEC has section prefix (e.g. 10804 → 0804)
            hs4_raw = str(rec.get("HS4 ID", ""))
            hs4     = hs4_raw[-4:].zfill(4) if len(hs4_raw) >= 4 else hs4_raw.zfill(4)

            rows.append({
                "hs_code_4digit":           hs4,
                "hs_code_6digit":           hs4 + "00",
                "product_name":             rec.get("HS4", ""),
                "hs4_oec_id":               rec.get("HS4 ID", ""),
                "reporter_code":            "12",
                "reporter_name":            "Algeria",
                "reporter_iso3":            "DZA",
                "reporter_oec_id":          rec.get(f"{exporter_lvl} ID", algeria_id),
                "partner_name":             rec.get(importer_lvl, ""),
                "partner_oec_id":           rec.get(f"{importer_lvl} ID", ""),
                "partner_code":             "",
                "partner_iso3":             "",
                "year":                     rec.get("Year", year),
                "period":                   rec.get("Year", year),
                "trade_value_usd":          float(trade_val) if trade_val else None,
                "net_weight_kg":            float(qty) * 1000 if qty else None,
                "gross_weight_kg":          None,
                "quantity_alt":             float(qty) if qty else None,
                "quantity_unit":            "tonnes",
                "quantity_unit_code":       "T",
                "unit_value_usd_per_kg":    round(float(trade_val) / (float(qty)*1000), 6)
                                            if trade_val and qty and float(qty) > 0 else None,
                "flow_code":                "X",
                "flow_desc":                "Exports",
                "classification":           cube,
                "aggregate_level":          "4",
                "is_leaf_code":             "",
                "customs_proc_code":        "",
                "mode_of_transport_code":   "",
                "data_source":              f"OEC_BACI_{cube}",
            })

        if not records or (offset + PAGE_SIZE) >= total:
            break

        offset += PAGE_SIZE
        page   += 1

    return rows


def fetch_all():
    progress = load_progress()
    done_years = set(progress.get("done_years", []))

    # ── Step 1: find working config ───────────────────────────────────────────
    cube         = progress.get("cube")
    exp_lvl      = progress.get("exporter_level")
    imp_lvl      = progress.get("importer_level")
    algeria_id   = progress.get("algeria_id")

    if not cube:
        cube, exp_lvl, imp_lvl, algeria_id = discover_working_config()
        if not cube:
            log.error(
                "\n❌ Could not find a working OEC API configuration.\n"
                "   The OEC may have changed their API schema.\n"
                "   Please register at comtradeplus.un.org for a free key\n"
                "   and paste it in the COMTRADE_KEY variable.\n"
                "   Or download manually from:\n"
                "   https://oec.world/en/profile/country/dza → Export tab → Download CSV"
            )
            return
        progress["cube"]            = cube
        progress["exporter_level"]  = exp_lvl
        progress["importer_level"]  = imp_lvl
        progress["algeria_id"]      = algeria_id
        save_progress(progress)
        log.info(f"\n✅ Config saved: {cube} | {exp_lvl} | {imp_lvl} | id={algeria_id}")
    else:
        log.info(f"✅ Using saved config: {cube} | {exp_lvl} | {imp_lvl} | id={algeria_id}")

    # ── Step 2: collect data year by year ─────────────────────────────────────
    all_rows = []
    out_path = Path("data/raw/12_algeria_exports_comtrade.csv")
    if out_path.exists():
        existing = pd.read_csv(out_path, low_memory=False)
        # Keep only rows from previous successful years
        existing = existing[existing["year"].isin(done_years)] if done_years else existing
        all_rows = existing.to_dict("records")
        log.info(f"Resuming — {len(all_rows):,} rows from completed years")

    for year in YEARS:
        if year in done_years:
            log.info(f"[SKIP] {year} already done")
            continue

        log.info(f"\n{'='*60}")
        log.info(f"Fetching year {year}")
        log.info(f"{'='*60}")

        year_rows = fetch_year(cube, exp_lvl, imp_lvl, algeria_id, year)
        log.info(f"  → {len(year_rows):,} rows for {year}")

        all_rows.extend(year_rows)
        done_years.add(year)
        progress["done_years"].append(year)
        save_progress(progress)

        pd.DataFrame(all_rows).to_csv(out_path, index=False)
        log.info(f"  💾 {len(all_rows):,} total rows saved")

    # ── Final summary ─────────────────────────────────────────────────────────
    if all_rows:
        df = pd.DataFrame(all_rows)
        df.to_csv(out_path, index=False)

        log.info(f"\n{'='*60}")
        log.info(f"✅ TABLE 12 COMPLETE")
        log.info(f"{'='*60}")
        log.info(f"  Rows              : {len(df):,}")
        log.info(f"  Destination ctrs  : {df['partner_name'].nunique()}")
        log.info(f"  HS4 products      : {df['hs_code_4digit'].nunique()}")
        log.info(f"  Years             : {sorted(df['year'].unique())}")
        log.info(f"  Total export USD  : ${df['trade_value_usd'].sum():,.0f}")

        top = (
            df.groupby("product_name")["trade_value_usd"]
            .sum().sort_values(ascending=False).head(15)
        )
        log.info("\n  Top 15 exports (all years combined):")
        for name, val in top.items():
            log.info(f"    {name:<45}  ${val:>15,.0f}")
    else:
        log.error("No data — see errors above")
        pd.DataFrame().to_csv(out_path, index=False)

    print("\n📁 data/raw/:")
    for f in sorted(Path("data/raw").glob("*.csv")):
        kb = f.stat().st_size / 1024
        try:
            n = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
        except Exception:
            n = 0
        print(f"  {f.name:<55} {n:>8,} rows  ({kb:,.0f} KB)")


if __name__ == "__main__":
    log.info("╔══════════════════════════════════════════════════════════════╗")
    log.info("║  TABLE 12 — Algeria Exports (OEC BACI, auto-probing schema)  ║")
    log.info("╚══════════════════════════════════════════════════════════════╝\n")

    # Clear old progress so it re-probes with fresh config
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        p["cube"] = None
        p["exporter_level"] = None
        p["importer_level"] = None
        p["algeria_id"] = None
        p["done_years"] = []
        save_progress(p)
        log.info("🧹 Progress reset — will re-probe schema\n")

    # Clear old page caches so they get re-fetched with correct params
    cleared = 0
    for f in Path("data/cache").glob("oec_v2_alg_*.json"):
        f.unlink()
        cleared += 1
    log.info(f"🧹 Cleared {cleared} old page cache files\n")

    fetch_all()

---
## ✅ Summary — Check All 12 Output Files
Run this cell after everything above to verify all tables were collected correctly.


In [ ]:
# Verify all 12 output files
from pathlib import Path

tables = [
    ("01_comtrade_world_imports.csv",       "Global imports — who buys what, from whom"),
    ("02_algeria_resources_fao.csv",        "Algeria production + area + yield"),
    ("03_algeria_resources_fao_inputs.csv", "Algeria fertilizers & inputs"),
    ("04_algeria_resources_fao_land.csv",   "Algeria land use & irrigation"),
    ("05_algeria_resources_fao_prices.csv", "Algeria domestic producer prices"),
    ("06_algeria_resources_worldbank.csv",  "Algeria resource rents (World Bank)"),
    ("07_fao_commodity_prices_monthly.csv", "World commodity prices monthly"),
    ("08_worldbank_commodity_index.csv",    "World Bank commodity price index"),
    ("09_comtrade_unit_values.csv",         "Implicit unit prices (value / weight)"),
    ("10_algeria_imports_comtrade.csv",     "Algeria own imports = supply gaps"),
    ("11_country_metadata.csv",            "All countries GDP, population, trade"),
    ("12_algeria_exports_comtrade.csv",     "Algeria exports by product & destination"),
]

print("=" * 70)
print("  DATA COLLECTION — OUTPUT FILE SUMMARY")
print("=" * 70)
all_ok = True
for fname, desc in tables:
    path = Path(f"data/raw/{fname}")
    if path.exists():
        rows = sum(1 for _ in open(path, encoding="utf-8", errors="ignore")) - 1
        kb   = path.stat().st_size / 1024
        icon = "✅" if rows > 0 else "⚠ "
        if rows == 0: all_ok = False
    else:
        rows, kb, icon = 0, 0, "❌"
        all_ok = False
    print(f" {icon}  {fname:<46}  {rows:>8,} rows  ({kb:>6.0f} KB)")
    print(f"       {desc}")
    print()

print("=" * 70)
if all_ok:
    print("🎉 All 12 tables ready — next: 02_data_preparation_eda.ipynb")
else:
    print("⚠  Some tables empty. Check logs/ folder and re-run failed cells.")
